<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:50px 40px;border-radius:12px;color:white;text-align:center;font-family:Arial,sans-serif;'>
  <p style='font-size:13px;letter-spacing:3px;color:#14BDBD;margin:0 0 8px 0;'>AI / ML FOUNDATIONS COHORT</p>
  <h1 style='font-size:38px;margin:0 0 8px 0;font-weight:900;'>MATH DEEP DIVE</h1>
  <h2 style='font-size:22px;font-weight:300;margin:0 0 30px 0;color:#D0D7E3;'>The Mathematical Engine Behind Every ML Model</h2>
  <div style='width:60px;height:3px;background:#F0A500;margin:0 auto 30px auto;'></div>
  <p style='font-size:14px;color:#D0D7E3;margin:0;'>Deep intuition · Proofs where they matter · Code throughout</p>
</div>


## What This Notebook Covers

| Section | Topic |
|---------|-------|
| 1 | Vectors, Dot Products & Matrix Multiplication |
| 2 | Probability Distributions — Normal, Binomial, and Beyond |
| 3 | Loss Functions — MSE, Cross-Entropy, Hinge and More |
| 4 | Gradient Descent — Built From Scratch, Step by Step |
| 5 | The Chain Rule & Backpropagation Intuition |
| 6 | Putting It All Together — A Neural Net From Pure Math |

> This notebook goes deeper than any previous session. Take your time. Run every cell. Break things deliberately.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,
                      'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
np.random.seed(42)
print('✅  Ready.')


---

# 🔢 Section 1: Vectors, Dot Products & Matrix Multiplication
#### *The language every ML model speaks*

---

### 1.1 What is a Vector?

A **vector** is an ordered list of numbers. In ML, every data point is a vector.
A house described by `[3 bedrooms, 120 sqm, 5 years, 2 bathrooms]` is a 4-dimensional vector.

Geometrically, a vector has **magnitude** (length) and **direction**.

$$\vec{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix} \qquad\|\vec{v}\| = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}$$

### 1.2 The Dot Product — The Most Important Operation in ML

The dot product between two vectors produces a **single number** (a scalar):

$$\vec{a} \cdot \vec{b} = \sum_{i=1}^{n} a_i b_i = a_1b_1 + a_2b_2 + \cdots + a_nb_n$$

**What it measures geometrically:**
$$\vec{a} \cdot \vec{b} = \|\vec{a}\| \cdot \|\vec{b}\| \cdot \cos(\theta)$$

- If $\theta = 0°$ (same direction): dot product is maximum — vectors are perfectly aligned
- If $\theta = 90°$ (perpendicular): dot product is 0 — vectors are completely unrelated
- If $\theta = 180°$ (opposite): dot product is negative — vectors point in opposite directions

> 🔑 **Why this matters in ML:** Every linear model prediction is a dot product.
> $\hat{y} = \vec{w} \cdot \vec{x} + b$ is literally the weights vector dotted with the features vector.


In [ ]:
# ── Vectors and the dot product from scratch ─────────────────────────────────
a = np.array([2.0, 3.0, 1.0, 4.0])   # feature vector (one data point)
w = np.array([0.5, 1.2, -0.3, 0.8])  # weight vector (what the model learned)
b = 1.5                                # bias

# Manual dot product
dot_manual = sum(ai * wi for ai, wi in zip(a, w))
dot_numpy  = np.dot(a, w)
prediction = np.dot(a, w) + b

print('=== Dot Product ===')
print(f'Features:    {a}')
print(f'Weights:     {w}')
print(f'Dot product (manual): {dot_manual:.4f}')
print(f'Dot product (numpy):  {dot_numpy:.4f}')
print(f'Prediction (w·x + b): {prediction:.4f}')

# Geometric interpretation — angle between vectors
cos_theta = np.dot(a, w) / (np.linalg.norm(a) * np.linalg.norm(w))
theta_deg = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
print(f'\nAngle between feature and weight vectors: {theta_deg:.1f}°')
print(f'Cosine similarity: {cos_theta:.4f}')
print('(Closer to 1 = more aligned = features match what model learned to look for)')


In [ ]:
# ── Visualising dot product geometrically (2D) ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [
    ([1,0], [1,0],  'Same direction\nDot product = max'),
    ([1,0], [0,1],  'Perpendicular\nDot product = 0'),
    ([1,0], [-1,0], 'Opposite\nDot product = min'),
]

for ax, (v1, v2, title) in zip(axes, pairs):
    ax.quiver(0,0,v1[0],v1[1], angles='xy', scale_units='xy', scale=1,
              color='#0D7377', width=0.05, label='Vector A')
    ax.quiver(0,0,v2[0],v2[1], angles='xy', scale_units='xy', scale=1,
              color='#F0A500', width=0.05, label='Vector B')
    dp = np.dot(v1, v2)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_title(f'{title}\n= {dp}', fontweight='bold', fontsize=10)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.legend(fontsize=8)

plt.suptitle('Dot Product: Geometric Interpretation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 1.3 Matrix Multiplication — Batched Predictions

When we have multiple data points (a batch), we stack them into a **matrix** and compute all predictions at once.

$$\hat{Y} = XW + b$$

Where:
- $X$ is the **data matrix** — shape `(n_samples, n_features)`
- $W$ is the **weight matrix** — shape `(n_features, n_outputs)`
- $\hat{Y}$ is the **predictions matrix** — shape `(n_samples, n_outputs)`

**The rule:** $(m \times k) \cdot (k \times n) = (m \times n)$ — inner dimensions must match.

$$C_{ij} = \sum_{k} A_{ik} \cdot B_{kj}$$


In [ ]:
# ── Matrix multiplication: from manual to numpy ───────────────────────────────

# 5 data points, 3 features each
X = np.array([
    [2.0, 1.5, 3.0],
    [1.0, 4.0, 2.0],
    [3.5, 0.5, 1.0],
    [0.5, 2.0, 4.5],
    [2.5, 3.0, 0.5],
])  # shape (5, 3)

W = np.array([
    [0.8,  0.3],
    [-0.5, 0.9],
    [0.2, -0.4],
])  # shape (3, 2) — 3 features in, 2 outputs

b = np.array([0.5, -0.2])  # bias for each output

# All 5 predictions in one operation
Y_hat = X @ W + b   # @ is Python's matrix multiply operator

print(f'X shape:     {X.shape}  (5 samples, 3 features)')
print(f'W shape:     {W.shape}  (3 features → 2 outputs)')
print(f'Y_hat shape: {Y_hat.shape}  (5 samples, 2 predictions each)')
print(f'\nPredictions:\n{Y_hat.round(3)}')
print('\nThis is exactly how a neural network layer computes its output —')
print('matrix multiply the input by the weights, add bias, apply activation.')


In [ ]:
# ✏️  YOUR TURN

# Create a (4, 5) matrix X and a (5, 3) weight matrix W
# Compute Y = X @ W
# Verify the output shape is (4, 3)
# Then: compute the magnitude (norm) of each row of Y
# Hint: np.linalg.norm(Y, axis=1)

# ── Your code here ───────────────────────────────────────────


---

# 🎲 Section 2: Probability Distributions
#### *The language of uncertainty — how models express confidence*

---

### 2.1 Why Probability?

ML models rarely give hard answers. They give **probabilities** — a confidence score for each possible outcome.
A spam classifier does not say 'this is spam.' It says 'I am 94.3% confident this is spam.'

Understanding the distributions that data follows tells you:
- Which model assumptions are valid
- How to initialise weights correctly
- How to interpret model outputs
- How to spot when something is wrong with your data

### 2.2 The Normal (Gaussian) Distribution

The most important distribution in ML. Defined by two parameters:
- **Mean** $\mu$ — the centre
- **Standard deviation** $\sigma$ — the spread

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

**Why it appears everywhere:**
- The Central Limit Theorem: averages of any distribution tend toward normal
- Neural network weights are usually initialised from a normal distribution
- Residuals (errors) in linear regression should be normally distributed
- Many natural phenomena follow it: height, measurement errors, noise


In [ ]:
# ── Normal Distribution: deep exploration ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

x = np.linspace(-5, 5, 1000)

# Effect of changing mean
for mu, color in [(-2,'#0D7377'), (0,'#F0A500'), (2,'#E74C3C')]:
    axes[0].plot(x, stats.norm.pdf(x, mu, 1), color=color, lw=2, label=f'μ={mu}, σ=1')
axes[0].set_title('Effect of Mean (μ)', fontweight='bold')
axes[0].legend(); axes[0].set_ylabel('Probability Density')

# Effect of changing std
for sigma, color in [(0.5,'#0D7377'), (1,'#F0A500'), (2,'#E74C3C')]:
    axes[1].plot(x, stats.norm.pdf(x, 0, sigma), color=color, lw=2, label=f'μ=0, σ={sigma}')
axes[1].set_title('Effect of Std Dev (σ)', fontweight='bold')
axes[1].legend()

# The 68-95-99.7 rule
x2 = np.linspace(-4, 4, 1000)
y2 = stats.norm.pdf(x2, 0, 1)
axes[2].plot(x2, y2, color='#1A2E4A', lw=2)
for lo, hi, color, label in [(-1,1,'#14BDBD','68%'), (-2,2,'#0D7377','95%'), (-3,3,'#1A2E4A22','99.7%')]:
    mask = (x2 >= lo) & (x2 <= hi)
    axes[2].fill_between(x2[mask], y2[mask], alpha=0.3, color=color, label=label)
axes[2].set_title('68-95-99.7 Rule', fontweight='bold')
axes[2].legend()

plt.suptitle('The Normal Distribution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Practical: checking if data is normal
sample = np.random.normal(loc=50, scale=10, size=1000)
stat, p_value = stats.shapiro(sample[:50])  # Shapiro-Wilk test
print(f'Shapiro-Wilk test: p-value = {p_value:.4f}')
print(f'Data is {"" if p_value > 0.05 else "NOT "}normally distributed (α=0.05)')


### 2.3 The Binomial Distribution

Models the number of successes in $n$ independent trials, each with probability $p$ of success.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**In ML context:**
- Bernoulli distribution ($n=1$): a single coin flip — binary classification output
- Logistic Regression output is a Bernoulli probability
- Dropout in neural networks uses Bernoulli sampling

### 2.4 Other Key Distributions

| Distribution | When It Appears in ML |
|-------------|----------------------|
| **Uniform** | Weight initialisation, random search, dropout masks |
| **Exponential** | Modelling time between events, Poisson processes |
| **Beta** | Modelling probabilities of probabilities — Bayesian ML |
| **Categorical** | Multi-class classification output (generalised Bernoulli) |
| **Dirichlet** | Prior over categorical distributions — topic modelling |


In [ ]:
# ── Binomial vs Normal — the Central Limit Theorem in action ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

p = 0.4
for i, n in enumerate([5, 20, 100]):
    k = np.arange(0, n+1)
    pmf = stats.binom.pmf(k, n, p)
    axes[i].bar(k, pmf, color='#0D7377', alpha=0.7, label='Binomial')
    # Overlay normal approximation
    mu_b  = n * p
    sig_b = np.sqrt(n * p * (1-p))
    x_c   = np.linspace(0, n, 300)
    axes[i].plot(x_c, stats.norm.pdf(x_c, mu_b, sig_b), color='#F0A500', lw=2, label='Normal approx')
    axes[i].set_title(f'n={n}, p={p}\nμ={mu_b:.1f}, σ={sig_b:.2f}', fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Central Limit Theorem: Binomial → Normal as n grows', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('As n increases, the binomial approaches a normal distribution.')
print('This is the Central Limit Theorem — one of the most powerful results in statistics.')


In [ ]:
# ── All key distributions side by side ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

x = np.linspace(-4, 4, 1000)
x_pos = np.linspace(0, 5, 1000)

plots = [
    ('Normal\n(weight init, noise)',   x,     stats.norm.pdf(x, 0, 1),              '#0D7377'),
    ('Uniform\n(random search)',        x,     stats.uniform.pdf(x, -4, 8),          '#F0A500'),
    ('Exponential\n(time between events)', x_pos, stats.expon.pdf(x_pos, scale=1),  '#E74C3C'),
    ('Beta(2,5)\n(Bayesian priors)',   np.linspace(0,1,1000), stats.beta.pdf(np.linspace(0,1,1000),2,5), '#8E44AD'),
    ('Chi-Squared\n(goodness of fit)', x_pos, stats.chi2.pdf(x_pos, df=3),          '#2E86AB'),
    ('Student t\n(small sample stats)',x,     stats.t.pdf(x, df=5),                 '#16A085'),
]

for ax, (title, xd, yd, color) in zip(axes, plots):
    ax.plot(xd, yd, color=color, lw=2)
    ax.fill_between(xd, yd, alpha=0.2, color=color)
    ax.set_title(title, fontweight='bold', fontsize=10)

plt.suptitle('Probability Distributions in ML Context', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ✏️  YOUR TURN

# 1. Generate 1000 samples from Normal(mu=70, sigma=15) — simulate exam scores
# 2. Plot the histogram with a KDE overlay
# 3. Calculate: what % of scores fall between 55 and 85?
#    Hint: use stats.norm.cdf(85, 70, 15) - stats.norm.cdf(55, 70, 15)
# 4. What score is at the 90th percentile?
#    Hint: stats.norm.ppf(0.90, 70, 15)
# 5. Run Shapiro-Wilk test to confirm normality

# ── Your code here ───────────────────────────────────────────


---

# 📉 Section 3: Loss Functions
#### *How models measure their own wrongness*

---

A **loss function** (also called cost function or objective function) quantifies how far a model's  
predictions are from the true values. Training is entirely about minimising this number.

Different problems need different loss functions. Choosing the wrong one is a common mistake.

### 3.1 Regression Losses

| Loss | Formula | When to Use | Sensitivity to Outliers |
|------|---------|-------------|------------------------|
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | Default for regression | High — squares large errors |
| **MAE** | $\frac{1}{n}\sum|y - \hat{y}|$ | When outliers exist | Low — treats all errors equally |
| **Huber** | MSE when small, MAE when large | Best of both worlds | Medium |
| **RMSE** | $\sqrt{MSE}$ | Same units as target | High |

### 3.2 Classification Losses

| Loss | Formula | When to Use |
|------|---------|-------------|
| **Binary Cross-Entropy** | $-[y\log\hat{p} + (1-y)\log(1-\hat{p})]$ | Binary classification |
| **Categorical Cross-Entropy** | $-\sum_c y_c \log\hat{p}_c$ | Multi-class classification |
| **Hinge Loss** | $\max(0, 1 - y \cdot \hat{y})$ | SVMs, margin-based models |
| **Focal Loss** | Modified cross-entropy for imbalanced classes | Object detection, rare classes |


In [ ]:
# ── Visualise regression losses ──────────────────────────────────────────────
residuals = np.linspace(-4, 4, 300)  # error = y_true - y_pred

mse   = residuals ** 2
mae   = np.abs(residuals)
delta = 1.0
huber = np.where(np.abs(residuals) <= delta,
                  0.5 * residuals**2,
                  delta * (np.abs(residuals) - 0.5 * delta))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(residuals, mse,   label='MSE',   color='#E74C3C', lw=2)
axes[0].plot(residuals, mae,   label='MAE',   color='#0D7377', lw=2)
axes[0].plot(residuals, huber, label='Huber', color='#F0A500', lw=2, linestyle='--')
axes[0].set_xlabel('Residual (y_true - y_pred)'); axes[0].set_ylabel('Loss')
axes[0].set_title('Regression Loss Functions', fontweight='bold')
axes[0].legend(); axes[0].set_ylim(0, 12)

# Gradients — how aggressively each loss pushes weights
grad_mse   = 2 * residuals
grad_mae   = np.sign(residuals)
grad_huber = np.where(np.abs(residuals) <= delta, residuals, delta * np.sign(residuals))

axes[1].plot(residuals, grad_mse,   label='MSE gradient',   color='#E74C3C', lw=2)
axes[1].plot(residuals, grad_mae,   label='MAE gradient',   color='#0D7377', lw=2)
axes[1].plot(residuals, grad_huber, label='Huber gradient', color='#F0A500', lw=2, linestyle='--')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Gradient (∂Loss/∂ŷ)')
axes[1].set_title('Gradients — How Hard Each Loss Pushes', fontweight='bold')
axes[1].legend(); axes[1].set_ylim(-5, 5)

plt.tight_layout(); plt.show()
print('Key insight: MSE gradient grows unboundedly — one large outlier can dominate training.')
print('Huber combines stability of MAE for large errors with smoothness of MSE for small ones.')


In [ ]:
# ── Binary Cross-Entropy — deep dive ─────────────────────────────────────────

# The formula: BCE = -[y*log(p) + (1-y)*log(1-p)]
# y = true label (0 or 1), p = predicted probability

def binary_cross_entropy(y_true, y_pred, eps=1e-15):
    p = np.clip(y_pred, eps, 1 - eps)  # prevent log(0)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

# Why cross-entropy? Compare with MSE for classification
p_range = np.linspace(0.01, 0.99, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss when true label = 1
bce_true1 = -np.log(p_range)           # loss when y=1
bce_true0 = -np.log(1 - p_range)       # loss when y=0
mse_true1 = (1 - p_range) ** 2
mse_true0 = p_range ** 2

axes[0].plot(p_range, bce_true1, color='#0D7377', lw=2, label='BCE (y=1)')
axes[0].plot(p_range, mse_true1, color='#F0A500', lw=2, linestyle='--', label='MSE (y=1)')
axes[0].set_title('Loss when True Label = 1', fontweight='bold')
axes[0].set_xlabel('Predicted Probability p'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].set_ylim(0, 5)
axes[0].annotate('BCE punishes\nconfident wrong\npredictions harder',
                  xy=(0.05, 3.0), fontsize=9, color='#0D7377')

axes[1].plot(p_range, bce_true0, color='#E74C3C', lw=2, label='BCE (y=0)')
axes[1].plot(p_range, mse_true0, color='#8E44AD', lw=2, linestyle='--', label='MSE (y=0)')
axes[1].set_title('Loss when True Label = 0', fontweight='bold')
axes[1].set_xlabel('Predicted Probability p'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].set_ylim(0, 5)

plt.suptitle('Cross-Entropy vs MSE for Classification', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Demonstrate
print('=== Why BCE > MSE for classification ===')
cases = [
    ('Model says 0.5 for y=1 (uncertain)', 1, 0.5),
    ('Model says 0.9 for y=1 (good!)',     1, 0.9),
    ('Model says 0.1 for y=1 (wrong!)',    1, 0.1),
    ('Model says 0.01 for y=1 (very wrong)',1,0.01),
]
print(f'{"Case":<40} {"BCE":>8} {"MSE":>8}')
print('-'*58)
for desc, y, p in cases:
    bce_val = binary_cross_entropy(np.array([y]), np.array([p]))
    mse_val = (y - p) ** 2
    print(f'{desc:<40} {bce_val:>8.4f} {mse_val:>8.4f}')
print('\nBCE punishes confident wrong predictions MUCH more aggressively than MSE.')
print('This creates a stronger training signal — the model learns faster from its worst mistakes.')


---

# ⛰️ Section 4: Gradient Descent
#### *Built from scratch — the engine of all learning*

---

**Gradient Descent** is the optimisation algorithm that trains virtually every ML model.
It answers one question: *given that I know how wrong I am right now, which direction should I adjust my weights?*

The answer comes from calculus. The **gradient** of the loss with respect to the weights tells us  
the direction of steepest increase. We go in the opposite direction — downhill.

$$w_{\text{new}} = w_{\text{old}} - \alpha \cdot \frac{\partial L}{\partial w}$$

Where $\alpha$ is the **learning rate** — how big a step we take each time.

### The Three Flavours

| Type | Data per update | Speed | Stability | When to use |
|------|----------------|-------|-----------|-------------|
| **Batch GD** | Full dataset | Slow | Very stable | Small datasets |
| **Stochastic GD (SGD)** | 1 sample | Fast | Noisy | Online learning |
| **Mini-batch GD** | Batch of 32-256 | Fast | Stable | Default for neural nets |


In [ ]:
# ── Gradient Descent from scratch on a simple problem ────────────────────────
# Problem: find the weight w that minimises Loss = (w - 3)^2
# True minimum: w = 3 (we'll let GD find it)

def loss_fn(w):        return (w - 3) ** 2
def gradient_fn(w):   return 2 * (w - 3)   # dL/dw

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lr in zip(axes, [0.01, 0.1, 0.9]):
    w = -2.0  # start far from optimum
    history_w, history_loss = [w], [loss_fn(w)]

    for step in range(50):
        grad = gradient_fn(w)
        w    = w - lr * grad
        history_w.append(w)
        history_loss.append(loss_fn(w))

    # Plot loss landscape
    w_range = np.linspace(-3, 7, 300)
    ax.plot(w_range, loss_fn(w_range), color='#1A2E4A', lw=2, label='Loss curve')
    ax.scatter(history_w, [loss_fn(wi) for wi in history_w],
               c=range(len(history_w)), cmap='YlOrRd', s=30, zorder=5)
    ax.scatter(history_w[0], loss_fn(history_w[0]), color='green', s=100, zorder=6, label='Start')
    ax.scatter(history_w[-1], loss_fn(history_w[-1]), color='red', s=100, zorder=6, label=f'End w={w:.3f}')
    ax.set_title(f'Learning Rate α={lr}', fontweight='bold')
    ax.set_xlabel('Weight w'); ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.set_ylim(-0.5, 30)

plt.suptitle('Gradient Descent with Different Learning Rates', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('Too small (0.01): converges correctly but slowly')
print('Just right (0.1): converges efficiently')
print('Too large (0.9):  overshoots and oscillates — may diverge!')


In [ ]:
# ── Full linear regression via gradient descent from scratch ─────────────────
np.random.seed(42)
n = 100
X_raw = np.random.uniform(0, 10, n)
y_raw = 2.5 * X_raw + 8.0 + np.random.normal(0, 3, n)  # true: w=2.5, b=8.0

# Normalise
X_norm = (X_raw - X_raw.mean()) / X_raw.std()

# Initialise weights
w, b   = 0.0, 0.0
lr     = 0.05
epochs = 200
losses = []
w_hist = [w]; b_hist = [b]

for epoch in range(epochs):
    # Forward pass
    y_hat = w * X_norm + b

    # Compute loss (MSE)
    loss = np.mean((y_raw - y_hat) ** 2)
    losses.append(loss)

    # Compute gradients
    dL_dw = -2 * np.mean((y_raw - y_hat) * X_norm)  # ∂L/∂w
    dL_db = -2 * np.mean(y_raw - y_hat)              # ∂L/∂b

    # Update weights
    w = w - lr * dL_dw
    b = b - lr * dL_db
    w_hist.append(w); b_hist.append(b)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Training curve
axes[0].plot(losses, color='#E74C3C', lw=2)
axes[0].set_title('Loss Over Training', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].axhline(min(losses), linestyle='--', color='gray', label=f'Min={min(losses):.2f}')
axes[0].legend()

# Weight convergence
axes[1].plot(w_hist, color='#0D7377', lw=2, label='w')
axes[1].plot(b_hist, color='#F0A500', lw=2, label='b')
axes[1].set_title('Weight Convergence', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Parameter value')
axes[1].legend()

# Final fit
axes[2].scatter(X_raw, y_raw, alpha=0.5, color='#0D7377', label='Data')
x_line = np.linspace(0, 10, 100)
x_line_norm = (x_line - X_raw.mean()) / X_raw.std()
axes[2].plot(x_line, w * x_line_norm + b, color='#F0A500', lw=2, label='GD fit')
axes[2].set_title('Final Model Fit', fontweight='bold')
axes[2].legend()

plt.tight_layout(); plt.show()
print(f'Learned:  w={w:.3f}, b={b:.3f}')
print(f'Final MSE: {losses[-1]:.4f}')
print('\nYou just implemented gradient descent from scratch — this is exactly what PyTorch does internally.')


In [ ]:
# ── Mini-batch vs SGD vs Batch GD comparison ─────────────────────────────────
np.random.seed(42)
n2 = 500
X2 = np.random.uniform(0, 10, n2)
y2 = 3.0 * X2 + 5.0 + np.random.normal(0, 4, n2)
X2_norm = (X2 - X2.mean()) / X2.std()

def run_gd(X, y, batch_size, lr=0.05, epochs=100):
    w, b, losses = 0.0, 0.0, []
    n = len(X)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        X_shuf, y_shuf = X[idx], y[idx]
        epoch_loss = []
        for start in range(0, n, batch_size):
            Xb = X_shuf[start:start+batch_size]
            yb = y_shuf[start:start+batch_size]
            yh = w * Xb + b
            loss = np.mean((yb - yh)**2)
            epoch_loss.append(loss)
            dw = -2*np.mean((yb-yh)*Xb)
            db = -2*np.mean(yb-yh)
            w -= lr*dw; b -= lr*db
        losses.append(np.mean(epoch_loss))
    return losses

batch_losses = run_gd(X2_norm, y2, batch_size=len(X2_norm))
mini_losses  = run_gd(X2_norm, y2, batch_size=32)
sgd_losses   = run_gd(X2_norm, y2, batch_size=1)

plt.figure(figsize=(10, 5))
plt.plot(batch_losses, label='Batch GD (full data)', color='#0D7377', lw=2)
plt.plot(mini_losses,  label='Mini-batch GD (size=32)', color='#F0A500', lw=2)
plt.plot(sgd_losses,   label='SGD (size=1)', color='#E74C3C', lw=2, alpha=0.7)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Batch vs Mini-batch vs SGD Loss Curves', fontweight='bold')
plt.legend()
plt.tight_layout(); plt.show()
print('SGD: noisiest but fastest to start moving')
print('Mini-batch: best of both — used in all deep learning')
print('Batch GD: smoothest curve but slowest per epoch on large data')


---

# 🔗 Section 5: The Chain Rule & Backpropagation
#### *How neural networks learn — the math behind the magic*

---

Backpropagation is the algorithm that computes gradients through a neural network.
It is nothing more than the **chain rule of calculus** applied repeatedly through each layer.

### 5.1 The Chain Rule

If $z = f(g(x))$, then:
$$\frac{dz}{dx} = \frac{dz}{dg} \cdot \frac{dg}{dx}$$

In words: the rate of change of the output with respect to the input equals the product  
of all the intermediate rates of change along the path.

### 5.2 Activation Functions and Their Derivatives

Activation functions introduce non-linearity — without them, a 100-layer network would collapse to a single linear equation.

| Activation | Formula | Derivative | Used In |
|-----------|---------|------------|--------|
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | $\sigma(x)(1-\sigma(x))$ | Output layer (binary) |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $1 - \tanh^2(x)$ | Hidden layers (older) |
| **ReLU** | $\max(0, x)$ | $0$ if $x<0$, $1$ if $x>0$ | Hidden layers (default) |
| **Leaky ReLU** | $\max(0.01x, x)$ | $0.01$ if $x<0$, $1$ if $x>0$ | Fixes dead neurons |
| **Softmax** | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | Complex Jacobian | Output layer (multi-class) |


In [ ]:
# ── Activation functions and their derivatives ────────────────────────────────
x = np.linspace(-4, 4, 300)

def sigmoid(x):      return 1 / (1 + np.exp(-x))
def tanh_fn(x):      return np.tanh(x)
def relu(x):         return np.maximum(0, x)
def leaky_relu(x):   return np.where(x >= 0, x, 0.01 * x)

def d_sigmoid(x):    s = sigmoid(x); return s * (1 - s)
def d_tanh(x):       return 1 - np.tanh(x)**2
def d_relu(x):       return (x > 0).astype(float)
def d_leaky(x):      return np.where(x >= 0, 1.0, 0.01)

funcs = [('Sigmoid', sigmoid, d_sigmoid, '#0D7377'),
          ('Tanh',    tanh_fn, d_tanh,    '#F0A500'),
          ('ReLU',    relu,    d_relu,    '#E74C3C'),
          ('Leaky ReLU', leaky_relu, d_leaky, '#8E44AD')]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for j, (name, fn, dfn, color) in enumerate(funcs):
    # Function
    axes[0, j].plot(x, fn(x), color=color, lw=2)
    axes[0, j].set_title(f'{name}\nf(x)', fontweight='bold')
    axes[0, j].axhline(0, color='gray', lw=0.5); axes[0, j].axvline(0, color='gray', lw=0.5)

    # Derivative
    axes[1, j].plot(x, dfn(x), color=color, lw=2, linestyle='--')
    axes[1, j].set_title(f'{name}\nf\'(x) — gradient', fontweight='bold')
    axes[1, j].axhline(0, color='gray', lw=0.5); axes[1, j].axvline(0, color='gray', lw=0.5)

plt.suptitle('Activation Functions and Their Derivatives', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('Key insight: The VANISHING GRADIENT PROBLEM')
print('  Sigmoid derivative max = 0.25. Tanh max = 1.0.')
print('  In deep networks, multiplying tiny gradients through many layers → near zero.')
print('  ReLU derivative = 1 for positive values — gradient flows much better.')
print('  This is why ReLU replaced sigmoid/tanh in hidden layers.')


In [ ]:
# ── Backpropagation: full worked example from scratch ────────────────────────
# A 2-layer neural network: input(2) → hidden(3) → output(1)
# Task: binary classification

np.random.seed(0)

# Tiny XOR-like dataset
X_bp = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_bp = np.array([[0],[1],[1],[0]], dtype=float)  # XOR

# Initialise weights (Xavier initialisation)
def xavier(fan_in, fan_out):
    limit = np.sqrt(6 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, (fan_in, fan_out))

W1 = xavier(2, 4); b1 = np.zeros((1, 4))
W2 = xavier(4, 1); b2 = np.zeros((1, 1))
lr = 0.5
losses_bp = []

for epoch in range(10000):
    # ── FORWARD PASS ─────────────────────────────────────────────
    Z1   = X_bp @ W1 + b1              # pre-activation layer 1
    A1   = np.tanh(Z1)                 # activation layer 1
    Z2   = A1 @ W2 + b2               # pre-activation layer 2
    A2   = sigmoid(Z2)                 # output (sigmoid for binary)

    # Loss (Binary Cross-Entropy)
    loss = -np.mean(y_bp * np.log(A2 + 1e-15) + (1-y_bp) * np.log(1-A2+1e-15))
    losses_bp.append(loss)

    # ── BACKWARD PASS (backpropagation) ──────────────────────────
    m    = X_bp.shape[0]

    dA2  = -(y_bp / (A2+1e-15)) + (1-y_bp)/(1-A2+1e-15)  # dL/dA2
    dZ2  = dA2 * A2 * (1 - A2)                             # dA2/dZ2 (sigmoid derivative)
    dW2  = A1.T @ dZ2 / m                                  # dZ2/dW2
    db2  = np.mean(dZ2, axis=0, keepdims=True)

    dA1  = dZ2 @ W2.T                                      # dZ2/dA1
    dZ1  = dA1 * (1 - np.tanh(Z1)**2)                     # dA1/dZ1 (tanh derivative)
    dW1  = X_bp.T @ dZ1 / m
    db1  = np.mean(dZ1, axis=0, keepdims=True)

    # ── UPDATE WEIGHTS ────────────────────────────────────────────
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1

print('=== Training Complete ===')
print(f'Final loss: {losses_bp[-1]:.6f}')
print(f'\nXOR Predictions:')
print(f'{"Input":<12} {"True":>6} {"Predicted":>12} {"Prob":>8}')
print('-'*42)
for xi, yi, pi in zip(X_bp, y_bp, A2):
    pred = 1 if pi[0] > 0.5 else 0
    print(f'{str(xi.astype(int)):<12} {int(yi[0]):>6} {pred:>12} {pi[0]:>8.4f}')

plt.figure(figsize=(7, 4))
plt.plot(losses_bp, color='#E74C3C', lw=1.5)
plt.title('Backprop Training — Loss Curve (XOR Problem)', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.yscale('log')
plt.tight_layout(); plt.show()
print('\nYou just implemented full backpropagation from scratch.')
print('Every deep learning framework (PyTorch, TensorFlow) does exactly this — automatically.')


---

# 🧠 Section 6: Everything Together — A Neural Net in Pure NumPy
#### *From math to a working model — no frameworks*

---

Now we build a proper, reusable neural network class using everything from this notebook.
This is not a toy — it is a clean, extensible implementation that will deepen your understanding  
of what PyTorch and TensorFlow are doing under the hood.


In [ ]:
class NeuralNetwork:
    """
    A flexible multi-layer neural network built from pure NumPy.
    Supports arbitrary depth, ReLU hidden activations, sigmoid output.
    """
    def __init__(self, layer_sizes, lr=0.01):
        self.lr = lr
        self.weights = []
        self.biases  = []
        for i in range(len(layer_sizes)-1):
            fan_in, fan_out = layer_sizes[i], layer_sizes[i+1]
            limit = np.sqrt(6 / (fan_in + fan_out))
            self.weights.append(np.random.uniform(-limit, limit, (fan_in, fan_out)))
            self.biases.append(np.zeros((1, fan_out)))

    def _relu(self, x):    return np.maximum(0, x)
    def _d_relu(self, x):  return (x > 0).astype(float)
    def _sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, X):
        self._activations = [X]
        self._pre_acts    = []
        A = X
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            Z = A @ W + b
            self._pre_acts.append(Z)
            A = self._sigmoid(Z) if i == len(self.weights)-1 else self._relu(Z)
            self._activations.append(A)
        return A

    def compute_loss(self, y_true, y_pred):
        p = np.clip(y_pred, 1e-15, 1-1e-15)
        return -np.mean(y_true*np.log(p) + (1-y_true)*np.log(1-p))

    def backward(self, X, y_true):
        m     = X.shape[0]
        A_out = self._activations[-1]
        delta = (A_out - y_true) / m
        for i in reversed(range(len(self.weights))):
            A_prev = self._activations[i]
            dW     = A_prev.T @ delta
            db     = np.sum(delta, axis=0, keepdims=True)
            if i > 0:
                delta = (delta @ self.weights[i].T) * self._d_relu(self._pre_acts[i-1])
            self.weights[i] -= self.lr * dW
            self.biases[i]  -= self.lr * db

    def train(self, X, y, epochs=1000, verbose=True):
        losses = []
        for epoch in range(epochs):
            y_hat = self.forward(X)
            loss  = self.compute_loss(y, y_hat)
            losses.append(loss)
            self.backward(X, y)
            if verbose and epoch % 200 == 0:
                acc = ((y_hat > 0.5).astype(int) == y).mean()
                print(f'  Epoch {epoch:>5} | Loss: {loss:.5f} | Acc: {acc:.2%}')
        return losses

    def predict(self, X, threshold=0.5):
        return (self.forward(X) > threshold).astype(int)

# ── Test on a real classification problem ────────────────────────────────────
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_m, y_m = make_moons(n_samples=500, noise=0.2, random_state=42)
y_m = y_m.reshape(-1, 1).astype(float)

scaler  = StandardScaler()
X_m     = scaler.fit_transform(X_m)
X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

np.random.seed(42)
net = NeuralNetwork(layer_sizes=[2, 16, 8, 1], lr=0.05)
print('Training a 3-layer neural network from scratch...')
losses_nn = net.train(X_tr, y_tr, epochs=1000, verbose=True)

acc = (net.predict(X_te) == y_te).mean()
print(f'\nTest accuracy: {acc:.2%}')


In [ ]:
# ── Visualise decision boundary ───────────────────────────────────────────────
h = 0.02
x_min, x_max = X_m[:,0].min()-0.5, X_m[:,0].max()+0.5
y_min, y_max = X_m[:,1].min()-0.5, X_m[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
Z = net.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.6)
axes[0].scatter(X_m[:,0], X_m[:,1], c=y_m.ravel(), cmap='RdYlGn',
                edgecolors='k', s=30)
axes[0].set_title('Decision Boundary — Pure NumPy Neural Net', fontweight='bold')

axes[1].plot(losses_nn, color='#E74C3C', lw=1.5)
axes[1].set_title('Training Loss Curve', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCE Loss')

plt.tight_layout(); plt.show()
print('This non-linear decision boundary is impossible for logistic regression.')
print('The network learned it purely from data, gradients, and the chain rule.')


In [ ]:
# ✏️  YOUR TURN

# EXTENDED CHALLENGE: Experiment with the NeuralNetwork class
# 
# 1. Create a deeper network: layer_sizes=[2, 32, 16, 8, 1]
#    Train it and compare test accuracy with the 3-layer version
# 
# 2. Try different learning rates: 0.001, 0.01, 0.1, 0.5
#    Plot all 4 loss curves on the same figure
# 
# 3. Try the make_circles dataset instead of make_moons:
#    from sklearn.datasets import make_circles
#    X_c, y_c = make_circles(n_samples=500, noise=0.1, random_state=42)
# 
# 4. Add an L2 regularisation term to the loss: loss += lambda * sum(W**2 for W in weights)
#    See if it reduces overfitting on a small dataset

# ── Your code here ───────────────────────────────────────────


---

## ✅ Math Deep Dive — Complete

| Topic | Covered |
|-------|--------|
| Vectors, dot products, matrix multiplication | ✅ |
| Normal, Binomial, and other distributions | ✅ |
| MSE, MAE, Huber, Cross-Entropy, Hinge losses | ✅ |
| Gradient descent from scratch (all 3 variants) | ✅ |
| Activation functions and their derivatives | ✅ |
| Backpropagation — fully implemented | ✅ |
| Full neural network in pure NumPy | ✅ |

> You now understand not just how to use ML tools — you understand *why* they work.
> Every framework you use from here is an abstraction over exactly what you built today.

<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:30px;border-radius:10px;color:white;text-align:center;'>
  <h3 style='margin:0 0 8px 0;color:#14BDBD;'>You built a neural network from pure mathematics.</h3>
  <p style='color:#F0A500;font-weight:bold;margin:10px 0 0 0;'>Most practitioners never go this deep. You did. 🚀</p>
</div>
